# 01 - Teacher cache

This notebook runs a larger teacher model over the prepared instruction-response records and stores top-k next-token logits. The student notebook uses this cache for KL-divergence distillation without repeatedly running the teacher during training.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


In [ ]:
@dataclass(frozen=True)
class TeacherConfig:
    teacher_model: str = "gpt2-medium"
    prepared_dir: Path = Path("../artifacts/prepared")
    cache_dir: Path = Path("../artifacts/teacher_cache")
    max_length: int = 384
    top_k: int = 32
    max_train_records: int | None = 256
    max_validation_records: int | None = 64


config = TeacherConfig()
config.cache_dir.mkdir(parents=True, exist_ok=True)
asdict(config)


In [ ]:
def select_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def read_jsonl(path: Path) -> list[dict]:
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]


def write_jsonl(records: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as handle:
        for record in records:
            handle.write(json.dumps(record) + "\n")


def limit_records(records: list[dict], limit: int | None) -> list[dict]:
    return records if limit is None else records[:limit]


In [ ]:
device = select_device()
dtype = torch.float16 if device.type == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(config.teacher_model, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

teacher = AutoModelForCausalLM.from_pretrained(
    config.teacher_model,
    torch_dtype=dtype,
)
teacher.to(device)
teacher.eval()

{"device": str(device), "vocab_size": len(tokenizer)}


In [ ]:
def response_loss_mask(prompt_ids: list[int], full_ids: list[int]) -> list[int]:
    response_start = min(len(prompt_ids), len(full_ids))
    return [int(position + 1 >= response_start) for position in range(max(len(full_ids) - 1, 0))]


def cache_record(record: dict) -> dict | None:
    prompt_ids = tokenizer(
        record["prompt"],
        add_special_tokens=False,
    )["input_ids"]
    encoded = tokenizer(
        record["text"],
        add_special_tokens=False,
        truncation=True,
        max_length=config.max_length,
        return_tensors="pt",
    )
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)
    if input_ids.shape[1] < 2:
        return None

    with torch.inference_mode():
        logits = teacher(input_ids=input_ids, attention_mask=attention_mask).logits[:, :-1, :]
        topk_values, topk_indices = torch.topk(logits[0], k=config.top_k, dim=-1)

    full_ids = input_ids[0].detach().cpu().tolist()
    loss_mask = response_loss_mask(prompt_ids, full_ids)
    if sum(loss_mask) == 0:
        return None

    return {
        "id": record["id"],
        "category": record.get("category", ""),
        "prompt": record["prompt"],
        "response": record["response"],
        "input_ids": full_ids,
        "attention_mask": attention_mask[0].detach().cpu().tolist(),
        "loss_mask": loss_mask,
        "teacher_topk_indices": topk_indices.detach().cpu().tolist(),
        "teacher_topk_logits": topk_values.detach().cpu().float().numpy().round(4).tolist(),
    }


In [ ]:
def build_split_cache(split: str, limit: int | None) -> dict:
    rows = limit_records(read_jsonl(config.prepared_dir / f"{split}.jsonl"), limit)
    cached = []
    skipped = 0
    for row in tqdm(rows, desc=f"teacher cache: {split}"):
        item = cache_record(row)
        if item is None:
            skipped += 1
        else:
            cached.append(item)

    output_path = config.cache_dir / f"{split}_teacher_topk.jsonl"
    write_jsonl(cached, output_path)
    return {"split": split, "written": len(cached), "skipped": skipped, "path": str(output_path)}


cache_summary = [
    build_split_cache("train", config.max_train_records),
    build_split_cache("validation", config.max_validation_records),
]
(config.cache_dir / "teacher_config.json").write_text(json.dumps(asdict(config), indent=2, default=str) + "\n")
cache_summary


In [ ]:
cached_preview = read_jsonl(config.cache_dir / "validation_teacher_topk.jsonl")[:2]
[
    {
        "id": row["id"],
        "tokens": len(row["input_ids"]),
        "supervised_tokens": sum(row["loss_mask"]),
        "top_k": len(row["teacher_topk_indices"][0]),
    }
    for row in cached_preview
]


In [ ]:
train_cache = read_jsonl(config.cache_dir / "train_teacher_topk.jsonl")
cache_stats = []
for row in train_cache:
    cache_stats.append({
        "tokens": len(row["input_ids"]),
        "supervised_tokens": sum(row["loss_mask"]),
        "topk_positions": len(row["teacher_topk_logits"]),
    })

cache_frame = pd.DataFrame(cache_stats)
cache_frame.describe().round(1)
